In [ ]:
import pandas as pd
import os

# Cada paciente tiene su propio esquema de nombres en QuPath, aqui se unifican
# todos a las mismas 4 categorias: tumor, stroma, stroma_linfos, unannotated

ANOTACIONES_DIR = "/home/imartinezle/Spatial_transcriptomics/Anotaciones_Patologos/"
COLUMNA_LABEL = "Layer"

# GIN71 es el unico caso donde "unknown" se decidio mapear a tumor (confirmado con Angel)
MAPEOS = {
    "GIN63": {
        "63 3T 4 tumor":  "tumor",
        "63 3T 4 stroma": "stroma",
        "Unannotated":    "unannotated",
    },
    "GIN65": {
        "tumor":          "tumor",
        "stroma":         "stroma",
        "stroma linfos":  "stroma_linfos",
        "Unannotated":    "unannotated",
    },
    "GIN67": {
        "tumor":          "tumor",
        "stroma":         "stroma",
        "stroma_linfos":  "stroma_linfos",
        "Unannotated":    "unannotated",
    },
    "GIN71": {
        "unknown":        "tumor",
        "stroma":         "stroma",
        "Unannotated":    "unannotated",
    },
}

ARCHIVOS = {
    "GIN63": os.path.join(ANOTACIONES_DIR, "PIT_17.csv"),
    "GIN65": os.path.join(ANOTACIONES_DIR, "PIT_20.csv"),
    "GIN67": os.path.join(ANOTACIONES_DIR, "PIT_22.csv"),
    "GIN71": os.path.join(ANOTACIONES_DIR, "PIT_28.csv"),
}

resumen = {}

for paciente, csv_path in ARCHIVOS.items():

    if not os.path.exists(csv_path):
        print(f"[{paciente}] no se encuentra {csv_path}, se omite")
        continue

    print(f"\nProcesando {paciente} ({os.path.basename(csv_path)})")

    df = pd.read_csv(csv_path, sep='\t')

    if COLUMNA_LABEL not in df.columns:
        print(f"  columna '{COLUMNA_LABEL}' no encontrada, columnas disponibles: {df.columns.tolist()}")
        continue

    print(f"  spots totales: {len(df)}")
    print(df[COLUMNA_LABEL].value_counts().to_string())

    mapeo = MAPEOS[paciente]
    df[COLUMNA_LABEL] = df[COLUMNA_LABEL].map(mapeo)

    # no deberia haber spots sin mapear, pero por si acaso quedan como unannotated
    sin_mapear = df[COLUMNA_LABEL].isna().sum()
    if sin_mapear > 0:
        print(f"  aviso: {sin_mapear} spots sin mapeo, se marcan como unannotated")
        df[COLUMNA_LABEL] = df[COLUMNA_LABEL].fillna("unannotated")

    print(f"  categorias normalizadas:")
    print(df[COLUMNA_LABEL].value_counts().to_string())

    output_path = os.path.join(ANOTACIONES_DIR, f"{paciente}_layer_normalizado.csv")
    df.to_csv(output_path, sep='\t', index=False)
    print(f"  guardado en {output_path}")

    resumen[paciente] = df[COLUMNA_LABEL].value_counts().to_dict()

print("\nResumen categorias normalizadas por paciente")
resumen_df = pd.DataFrame(resumen).fillna(0).astype(int)
print(resumen_df)